Codigo final del sistema de limpieza y creacion de dataset para el modelo de IA

In [ ]:
"""
================================================================================
  REMMAQ — Dataset ML Builder  |  Interfaz Gradio
  Autor : Senior Data Engineer
  Uso   : python remmaq_gradio.py
================================================================================
  Instalación de dependencias:
      pip install gradio pandas numpy openpyxl xlrd
================================================================================
"""

import io
import os
import shutil
import tempfile
import warnings
import numpy as np
import pandas as pd
import gradio as gr
from pathlib import Path

warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════════════════════════════════════
# CONSTANTES
# ══════════════════════════════════════════════════════════════════════════════

CONTAMINANTES: list[str] = ["PM25", "PM10", "O3", "CO", "NO2", "SO2"]
TARGET: str = "PM25"
PANDEMIA_INICIO = "2020-01-01"
PANDEMIA_FIN    = "2021-12-31"

RANGOS_VALIDOS: dict = {
    "PM25":             (0, 400),
    "PM10":             (0, 999),
    "O3":               (0, 500),
    "CO":               (0, 50),
    "NO2":              (0, 500),
    "SO2":              (0, 500),
    "Temperatura":      (-10, 50),
    "Humedad":          (0, 100),
    "Viento_Velocidad": (0, 50),
    "Viento_Direccion": (0, 360),
    "Precipitacion":    (0, 200),
}

NOMBRES_ARCHIVOS: dict = {
    "PM25":             ["PM2.5.xlsx", "PM25.xlsx", "PM25.csv"],
    "PM10":             ["PM10.xlsx",  "PM10.csv"],
    "O3":               ["O3.xlsx",    "O3.csv"],
    "CO":               ["CO.xlsx",    "CO.csv"],
    "NO2":              ["NO2.xlsx",   "NO2.csv"],
    "SO2":              ["SO2.xlsx",   "SO2.csv"],
    "Temperatura":      ["TMP.xlsx",   "Temperatura.xlsx", "Temperatura.csv"],
    "Humedad":          ["HUM.xlsx",   "Humedad.xlsx",     "Humedad.csv"],
    "Viento_Velocidad": ["VEL.xlsx",   "Viento_Velocidad.xlsx"],
    "Viento_Direccion": ["DIR.xlsx",   "Viento_Direccion.xlsx"],
    "Precipitacion":    ["LLU.xlsx",   "Precipitacion.xlsx"],
}

ALL_EXPECTED_NAMES = [n for nlist in NOMBRES_ARCHIVOS.values() for n in nlist]


# ══════════════════════════════════════════════════════════════════════════════
# LÓGICA DE PROCESAMIENTO
# ══════════════════════════════════════════════════════════════════════════════

def _leer_archivo(path: Path) -> pd.DataFrame:
    ext = path.suffix.lower()
    if ext == ".csv":
        for sep in (",", ";", "\t"):
            try:
                df = pd.read_csv(path, sep=sep, low_memory=False)
                if df.shape[1] > 1:
                    return df
            except Exception:
                continue
        raise ValueError(f"No se pudo leer el CSV: {path}")
    elif ext in (".xlsx", ".xls"):
        return pd.read_excel(path)
    raise ValueError(f"Extensión no soportada: {ext}")


def _preparar_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.rename(columns={df.columns[0]: "Fecha"})
    mask_u = df["Fecha"].astype(str).str.contains(
        r"unidad|unit|ug|mg|%|m/s|°|grados", case=False, na=False, regex=True
    )
    df = df[~mask_u].reset_index(drop=True)
    df["Fecha"] = pd.to_datetime(df["Fecha"], errors="coerce")
    df = df.dropna(subset=["Fecha"]).set_index("Fecha").sort_index()
    df.columns = [str(c).strip() for c in df.columns]
    return df


def buscar_archivos(data_dir: Path) -> dict[str, Path]:
    encontrados = {}
    for variable, candidatos in NOMBRES_ARCHIVOS.items():
        for nombre in candidatos:
            ruta = data_dir / nombre
            if ruta.exists():
                encontrados[variable] = ruta
                break
    return encontrados


def obtener_parroquias(archivos: dict[str, Path]) -> list[str]:
    for var in ["PM25", "PM10", "O3", "CO", "NO2", "SO2"]:
        if var in archivos:
            try:
                df = _leer_archivo(archivos[var])
                df = _preparar_df(df)
                cols = [c for c in df.columns if c.upper() not in ("FECHA", "DATE")]
                if cols:
                    return sorted([c.upper() for c in cols])
            except Exception:
                continue
    return []


def _extraer_columna(path: Path, variable: str, parroquia: str, log: list) -> pd.Series:
    if not path.exists():
        log.append(f"  ✗  [{variable:<22}] No encontrado: {path.name}")
        return pd.Series(dtype=float, name=variable)

    df = _leer_archivo(path)
    df = _preparar_df(df)

    pu = parroquia.strip().upper()
    col_match = None
    for col in df.columns:
        if col.upper() == pu:
            col_match = col
            break
    if col_match is None:
        candidatos = [c for c in df.columns if pu in c.upper()]
        if candidatos:
            col_match = candidatos[0]
            log.append(f"  ⚠  [{variable:<22}] parcial → '{col_match}'")
    if col_match is None:
        log.append(f"  ✗  [{variable:<22}] '{parroquia}' no encontrada")
        return pd.Series(dtype=float, name=variable)

    serie = pd.to_numeric(df[col_match], errors="coerce")
    serie.name = variable
    n_val = serie.notna().sum()
    rango = f"{serie.index.min().date()} → {serie.index.max().date()}"
    log.append(f"  ✓  [{variable:<22}] {n_val:>8,} valores  |  {rango}")
    return serie


def construir_dataset(
    archivos: dict[str, Path],
    parroquia: str,
    excluir_pandemia: bool,
    umbral_gap: int,
) -> tuple[pd.DataFrame, str]:
    log: list[str] = []
    SEP = "═" * 62

    log.append(SEP)
    log.append(f"  REMMAQ · Dataset ML — Parroquia: {parroquia.upper()}")
    log.append(f"  Target: {TARGET}  |  Gap: {umbral_gap}h  |  Pandemia excluida: {excluir_pandemia}")
    log.append(SEP)
    log.append("")

    # Cargar series
    log.append("▶ PASO 1 · Cargando archivos")
    series = []
    for var, ruta in archivos.items():
        s = _extraer_columna(ruta, var, parroquia, log)
        if not s.empty:
            series.append(s)

    if not series:
        raise RuntimeError(f"No se encontró '{parroquia}' en ningún archivo.")

    df = pd.concat(series, axis=1, join="outer").sort_index()
    df.index.name = "Timestamp"
    log.append(f"\n  Dataset bruto: {df.shape[0]:,} filas × {df.shape[1]} columnas")
    log.append(f"  Período: {df.index.min()} → {df.index.max()}")

    n_total = len(df)
    etapas = []

    def reg(etapa, n_a, n_d):
        elim = n_a - n_d
        pct = elim / n_total * 100 if n_total else 0
        etapas.append((etapa, elim))
        ico = "✂" if elim else "✓"
        log.append(f"  {ico}  {etapa:<46}  −{elim:>7,} ({pct:.1f}%)")

    # Limpiar sensores
    log.append("\n▶ PASO 2 · Limpiando errores de sensor")
    df_c = df.copy()
    n_errores = 0
    for col in df_c.columns:
        n0 = df_c[col].notna().sum()
        df_c.loc[df_c[col].isin([-999, -9999, 9999]), col] = np.nan
        rng = RANGOS_VALIDOS.get(col)
        if rng:
            vmin, vmax = rng
            df_c.loc[(df_c[col] < vmin) | (df_c[col] > vmax), col] = np.nan
        n_errores += n0 - df_c[col].notna().sum()
    df = df_c
    log.append(f"  ✓  {n_errores:,} valores anómalos → NaN")

    # Filtros
    log.append("\n▶ PASO 3 · Filtros de coherencia ML")

    if excluir_pandemia:
        n = len(df)
        mask = (df.index >= PANDEMIA_INICIO) & (df.index <= PANDEMIA_FIN)
        df = df[~mask].copy()
        reg(f"Pandemia ({PANDEMIA_INICIO}→{PANDEMIA_FIN})", n, len(df))

    # R1
    if TARGET not in df.columns:
        raise RuntimeError(f"Columna '{TARGET}' no encontrada. ¿Subiste PM2.5.xlsx o PM25.xlsx?")
    n = len(df)
    df = df.dropna(subset=[TARGET]).copy()
    reg("R1 — Sin target (PM25=NaN)", n, len(df))

    # R2
    cols = [c for c in CONTAMINANTES if c in df.columns]
    if len(cols) >= 2:
        n = len(df)
        df = df[df[cols].notna().any(axis=1)].copy()
        reg("R2 — Sin ningún contaminante (falla total)", n, len(df))

    # R3
    if cols:
        n = len(df)
        silencio = df[cols].isna().all(axis=1)
        cambio = silencio != silencio.shift()
        id_blq = cambio.cumsum()
        df_sil = df[silencio].copy()
        blqs_malos: set = set()
        if not df_sil.empty:
            df_sil["_blq_"] = id_blq[silencio]
            durs = df_sil.groupby("_blq_").apply(
                lambda g: (g.index.max() - g.index.min()).total_seconds() / 3600
            )
            blqs_malos = set(durs[durs > umbral_gap].index.tolist())
            if blqs_malos:
                log.append(f"\n  Gaps detectados (>{umbral_gap}h):")
                for bid in sorted(blqs_malos):
                    g2 = df_sil[df_sil["_blq_"] == bid]
                    log.append(f"    • {g2.index.min()} → {g2.index.max()} ({durs[bid]:.1f}h)")
            mascara = silencio & id_blq.isin(blqs_malos)
            df = df[~mascara].copy()
        reg(f"R3 — Gaps >{umbral_gap}h", n, len(df))

    # Resumen filtrado
    n_final = len(df)
    ret = n_final / n_total * 100 if n_total else 0
    log.append(f"\n{'─'*62}")
    log.append(f"  {'Filas originales':<46}  {n_total:>10,}")
    for e, elim in etapas:
        log.append(f"  {e:<46}  −{elim:>9,}")
    log.append(f"{'─'*62}")
    log.append(f"  {'Filas finales':<46}  {n_final:>10,}  ({ret:.1f}%)")
    log.append(SEP)

    if df.empty:
        raise RuntimeError("El dataset quedó vacío tras el filtrado.")

    # Validación
    assert df[TARGET].isna().sum() == 0, "FALLO: PM25 tiene NaN residuales."

    # Features ML
    log.append("\n▶ PASO 4 · Generando features ML")
    df["PM25_lag_1h"]  = df[TARGET].shift(1)
    df["PM25_lag_3h"]  = df[TARGET].shift(3)
    df["PM25_lag_24h"] = df[TARGET].shift(24)
    hour  = df.index.hour
    month = df.index.month
    df["hora_sin"] = np.sin(2 * np.pi * hour / 24)
    df["hora_cos"] = np.cos(2 * np.pi * hour / 24)
    df["mes_sin"]  = np.sin(2 * np.pi * month / 12)
    df["mes_cos"]  = np.cos(2 * np.pi * month / 12)
    log.append(f"  ✓  +7 columnas (lags PM25 + cíclicas hora/mes)")
    log.append(f"  ✓  Total columnas: {df.shape[1]}")
    log.append(f"\n  ✅  Dataset listo — {len(df):,} filas · {df.shape[1]} columnas")
    log.append(SEP + "\n")

    return df, "\n".join(log)


def df_a_csv(df: pd.DataFrame, parroquia: str) -> str:
    meta = [
        "# ═══════════════════════════════════════════════════════════════",
        "# DATASET ML — Red REMMAQ / Quito",
        f"# Parroquia    : {parroquia.upper()}",
        f"# Target       : PM25 (µg/m³)",
        f"# Filas        : {len(df):,}",
        f"# Columnas     : {df.shape[1]}  →  {list(df.columns)}",
        f"# Inicio       : {df.index.min()}",
        f"# Fin          : {df.index.max()}",
        f"# Generado     : {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}",
        "#",
        "# Uso: pd.read_csv('...csv', comment='#', index_col='Timestamp', parse_dates=True)",
        "# ═══════════════════════════════════════════════════════════════",
    ]
    buf = io.StringIO()
    for line in meta:
        buf.write(line + "\n")
    df.to_csv(buf, date_format="%Y-%m-%d %H:%M")
    return buf.getvalue()


# ══════════════════════════════════════════════════════════════════════════════
# ESTADO GLOBAL (sesión)
# ══════════════════════════════════════════════════════════════════════════════

_estado = {
    "archivos": {},
    "parroquias": [],
    "temp_dir": None,
}


# ══════════════════════════════════════════════════════════════════════════════
# FUNCIONES DE CALLBACK
# ══════════════════════════════════════════════════════════════════════════════

def escanear_directorio(directorio: str):
    """Escanea el directorio y actualiza el estado."""
    data_dir = Path(directorio.strip())
    if not data_dir.exists():
        return (
            gr.update(value=f"❌ El directorio `{directorio}` no existe.", visible=True),
            gr.update(choices=[], value=None),
            gr.update(value=_tabla_archivos({})),
        )

    archivos = buscar_archivos(data_dir)
    if not archivos:
        return (
            gr.update(
                value=f"⚠️ No se encontraron archivos REMMAQ en `{directorio}`.\n"
                       "Usa la pestaña de **Subida manual**.",
                visible=True,
            ),
            gr.update(choices=[], value=None),
            gr.update(value=_tabla_archivos({})),
        )

    _estado["archivos"] = archivos
    parroquias = obtener_parroquias(archivos)
    _estado["parroquias"] = parroquias

    msg = (
        f"✅ {len(archivos)} archivo(s) encontrado(s): "
        + ", ".join(archivos.keys())
    )
    return (
        gr.update(value=msg, visible=True),
        gr.update(choices=parroquias, value=parroquias[0] if parroquias else None),
        gr.update(value=_tabla_archivos(archivos)),
    )


def procesar_subida(files):
    """Guarda archivos subidos en un temp dir y escanea."""
    if not files:
        return (
            gr.update(value="⚠️ No se subieron archivos.", visible=True),
            gr.update(choices=[], value=None),
            gr.update(value=_tabla_archivos({})),
        )

    if _estado["temp_dir"] is None:
        _estado["temp_dir"] = tempfile.mkdtemp()
    tmp = Path(_estado["temp_dir"])

    for f in files:
        src = Path(f.name)
        dest = tmp / src.name
        shutil.copy2(src, dest)

    archivos = buscar_archivos(tmp)
    _estado["archivos"] = archivos

    if not archivos:
        no_reconocidos = [Path(f.name).name for f in files]
        return (
            gr.update(
                value=f"❌ Ningún archivo reconocido. Nombres esperados: "
                      f"PM2.5.xlsx, PM10.xlsx, SO2.xlsx, etc. "
                      f"Recibidos: {', '.join(no_reconocidos)}",
                visible=True,
            ),
            gr.update(choices=[], value=None),
            gr.update(value=_tabla_archivos({})),
        )

    parroquias = obtener_parroquias(archivos)
    _estado["parroquias"] = parroquias

    msg = f"✅ {len(archivos)} archivo(s) reconocido(s): " + ", ".join(archivos.keys())
    return (
        gr.update(value=msg, visible=True),
        gr.update(choices=parroquias, value=parroquias[0] if parroquias else None),
        gr.update(value=_tabla_archivos(archivos)),
    )


def _tabla_archivos(archivos: dict) -> pd.DataFrame:
    rows = []
    for var, nlist in NOMBRES_ARCHIVOS.items():
        encontrado = var in archivos
        rows.append({
            "Variable": var,
            "Estado": "✓ Cargado" if encontrado else "✗ No encontrado",
            "Archivo": archivos[var].name if encontrado else "—",
        })
    return pd.DataFrame(rows)


def ejecutar_pipeline(
    parroquia: str,
    excluir_pandemia: bool,
    umbral_gap: int,
    progress=gr.Progress(track_tqdm=True),
):
    """Ejecuta el pipeline y retorna log, métricas, preview y ruta del CSV."""
    archivos = _estado.get("archivos", {})
    if not archivos:
        return (
            "❌ No hay archivos cargados. Primero escanea un directorio o sube archivos.",
            None,
            None,
            None,
        )
    if not parroquia:
        return (
            "❌ Selecciona o escribe una parroquia primero.",
            None,
            None,
            None,
        )

    try:
        progress(0.1, desc="Cargando archivos…")
        df, log_text = construir_dataset(
            archivos=archivos,
            parroquia=parroquia.strip().upper(),
            excluir_pandemia=excluir_pandemia,
            umbral_gap=int(umbral_gap),
        )
        progress(0.9, desc="Exportando CSV…")

        # Guardar CSV temporal
        tmp_out = tempfile.NamedTemporaryFile(
            delete=False,
            suffix=".csv",
            prefix=f"dataset_ml_{parroquia.lower().replace(' ','_')}_",
        )
        contenido = df_a_csv(df, parroquia)
        tmp_out.write(contenido.encode("utf-8"))
        tmp_out.close()

        # Perfil para mostrar
        perfil_rows = []
        for col in df.select_dtypes(include=[np.number]).columns:
            perfil_rows.append({
                "Variable": col,
                "N válidos": int(df[col].notna().sum()),
                "% NaN": round(df[col].isna().mean() * 100, 1),
                "Media": round(df[col].mean(), 3),
                "Mín": round(df[col].min(), 3),
                "Máx": round(df[col].max(), 3),
            })

        progress(1.0, desc="¡Listo!")
        return (
            log_text,
            pd.DataFrame(perfil_rows),
            df.head(100),
            tmp_out.name,
        )

    except Exception as e:
        return (str(e), None, None, None)


# ══════════════════════════════════════════════════════════════════════════════
# TEMA Y CSS PERSONALIZADO
# ══════════════════════════════════════════════════════════════════════════════

CSS = """
@import url('https://fonts.googleapis.com/css2?family=Space+Mono:wght@400;700&family=DM+Sans:wght@300;400;600;700&display=swap');

/* ── Base ── */
body, .gradio-container {
    font-family: 'DM Sans', sans-serif !important;
    background: #f0f4f8 !important;
}
.dark body, .dark .gradio-container {
    background: #0f1923 !important;
}

/* ── Header ── */
.remmaq-title {
    background: linear-gradient(135deg, #1a2b3c 0%, #0f4c75 100%);
    color: white;
    padding: 2rem 2.5rem;
    border-radius: 16px;
    margin-bottom: 1.5rem;
    border-left: 5px solid #00d4aa;
    box-shadow: 0 8px 32px rgba(0,212,170,0.15);
}
.remmaq-title h1 {
    font-family: 'DM Sans', sans-serif !important;
    font-size: 2rem !important;
    font-weight: 700 !important;
    margin: 0 0 0.3rem !important;
    color: #00d4aa !important;
}
.remmaq-title p {
    color: #a8c7e0 !important;
    margin: 0 !important;
    font-size: 0.9rem !important;
}

/* ── Tabs ── */
.tab-nav button {
    font-family: 'DM Sans', sans-serif !important;
    font-weight: 600 !important;
    font-size: 0.9rem !important;
    border-radius: 8px 8px 0 0 !important;
    padding: 0.6rem 1.2rem !important;
}
.tab-nav button.selected {
    background: #0f4c75 !important;
    color: #00d4aa !important;
    border-bottom: 3px solid #00d4aa !important;
}

/* ── Botón principal ── */
#btn-ejecutar {
    background: linear-gradient(135deg, #00d4aa, #0f4c75) !important;
    color: white !important;
    font-family: 'DM Sans', sans-serif !important;
    font-weight: 700 !important;
    font-size: 1rem !important;
    border: none !important;
    border-radius: 10px !important;
    padding: 0.75rem 2rem !important;
    box-shadow: 0 4px 15px rgba(0,212,170,0.3) !important;
    transition: all 0.25s ease !important;
}
#btn-ejecutar:hover {
    transform: translateY(-2px) !important;
    box-shadow: 0 6px 20px rgba(0,212,170,0.45) !important;
}

/* ── Botón secundario ── */
#btn-escanear, #btn-subida {
    background: #1a2b3c !important;
    color: #00d4aa !important;
    border: 1.5px solid #00d4aa !important;
    border-radius: 8px !important;
    font-weight: 600 !important;
    transition: all 0.2s !important;
}
#btn-escanear:hover, #btn-subida:hover {
    background: #00d4aa !important;
    color: #0f1923 !important;
}

/* ── Log console ── */
#log-output textarea {
    font-family: 'Space Mono', monospace !important;
    font-size: 0.78rem !important;
    background: #0d1b2a !important;
    color: #00d4aa !important;
    border: 1px solid #1e3a5f !important;
    border-radius: 10px !important;
    line-height: 1.7 !important;
}

/* ── Inputs ── */
input[type="text"], input[type="number"] {
    border-radius: 8px !important;
    border: 1.5px solid #cbd5e1 !important;
    font-family: 'DM Sans', sans-serif !important;
}

/* ── Status ── */
#status-box textarea {
    border-radius: 10px !important;
    font-family: 'DM Sans', sans-serif !important;
    font-weight: 500 !important;
}

/* ── Dataframe ── */
.svelte-1gfkzm4, table {
    font-family: 'DM Sans', sans-serif !important;
    font-size: 0.83rem !important;
}

/* ── Card info ── */
.info-card {
    background: linear-gradient(135deg, #e8f4fd, #f0fdf9);
    border: 1px solid #bde0fe;
    border-left: 4px solid #00d4aa;
    border-radius: 10px;
    padding: 0.9rem 1.2rem;
    font-size: 0.85rem;
    color: #1a2b3c;
    margin-bottom: 0.75rem;
}

/* ── Accordion ── */
details summary {
    font-family: 'DM Sans', sans-serif !important;
    font-weight: 600 !important;
    color: #1a2b3c !important;
}
"""

THEME = gr.themes.Base(
    primary_hue="cyan",
    secondary_hue="blue",
    neutral_hue="slate",
    font=[gr.themes.GoogleFont("DM Sans"), "sans-serif"],
).set(
    body_background_fill="#f0f4f8",
    block_background_fill="#ffffff",
    block_border_color="#e2e8f0",
    block_radius="12px",
    block_shadow="0 2px 12px rgba(0,0,0,0.06)",
    button_primary_background_fill="#00d4aa",
    button_primary_text_color="#0f1923",
    button_primary_border_color="#00d4aa",
    input_background_fill="#f8fafc",
)


# ══════════════════════════════════════════════════════════════════════════════
# CONSTRUCCIÓN DE LA INTERFAZ
# ══════════════════════════════════════════════════════════════════════════════

def build_app() -> gr.Blocks:
    with gr.Blocks(
        theme=THEME,
        css=CSS,
        title="REMMAQ · Dataset ML Builder",
    ) as app:

        # ── Header ────────────────────────────────────────────────────────────
        gr.HTML("""
        <div class="remmaq-title">
          <h1>🌫️ REMMAQ · Dataset ML Builder</h1>
          <p>Red Metropolitana de Monitoreo Atmosférico de Quito
             &nbsp;·&nbsp; Genera datasets listos para
             <code>HistGradientBoostingRegressor</code></p>
        </div>
        """)

        # ══════════════════════════════════════════════════════════════════════
        # FILA PRINCIPAL
        # ══════════════════════════════════════════════════════════════════════
        with gr.Row():

            # ── COLUMNA IZQUIERDA: Carga + Config ─────────────────────────────
            with gr.Column(scale=4):

                with gr.Tabs():

                    # ── Tab 1: Directorio ────────────────────────────────────
                    with gr.TabItem("📁 Directorio local"):
                        gr.Markdown(
                            "Ingresa la ruta donde están los archivos REMMAQ "
                            "(`.xlsx` / `.csv`) y pulsa **Escanear**."
                        )
                        with gr.Row():
                            txt_dir = gr.Textbox(
                                label="Ruta del directorio",
                                placeholder="/ruta/a/tus/archivos/remmaq",
                                scale=4,
                            )
                            btn_scan = gr.Button("🔍 Escanear", elem_id="btn-escanear", scale=1)

                    # ── Tab 2: Subida manual ─────────────────────────────────
                    with gr.TabItem("⬆️ Subir archivos"):
                        gr.Markdown(
                            "Sube los archivos REMMAQ. El nombre del archivo "
                            "debe coincidir con los esperados: "
                            "`PM2.5.xlsx`, `PM10.xlsx`, `SO2.xlsx`, `CO.xlsx`, "
                            "`O3.xlsx`, `NO2.xlsx`, `TMP.xlsx`, `HUM.xlsx`, "
                            "`VEL.xlsx`, `DIR.xlsx`, `LLU.xlsx`"
                        )
                        file_upload = gr.File(
                            label="Archivos REMMAQ",
                            file_count="multiple",
                            file_types=[".xlsx", ".xls", ".csv"],
                        )
                        btn_upload = gr.Button(
                            "📂 Procesar archivos subidos",
                            elem_id="btn-subida",
                        )

                # Estado
                status_box = gr.Textbox(
                    label="Estado",
                    interactive=False,
                    elem_id="status-box",
                    max_lines=3,
                    visible=False,
                )

                # Tabla de archivos detectados
                with gr.Accordion("📋 Archivos detectados", open=False):
                    tabla_archivos = gr.Dataframe(
                        value=_tabla_archivos({}),
                        interactive=False,
                        wrap=True,
                    )

                gr.Markdown("---")

                # ── Configuración ────────────────────────────────────────────
                gr.Markdown("### ⚙️ Configuración")

                parroquia_dd = gr.Dropdown(
                    label="Parroquia / Estación",
                    choices=[],
                    allow_custom_value=True,
                    info="Se detecta automáticamente al cargar archivos, o escribe el nombre",
                )

                excluir_pandemia = gr.Checkbox(
                    label="Excluir período pandemia COVID-19 (2020–2021)",
                    value=True,
                    info="Elimina datos 2020-01-01 → 2021-12-31 para evitar distorsiones",
                )

                umbral_gap = gr.Slider(
                    label="Umbral de gap (horas)",
                    minimum=6, maximum=72, value=24, step=6,
                    info="Elimina bloques donde todos los sensores están en silencio > N horas",
                )

                btn_run = gr.Button(
                    "▶  Construir Dataset ML",
                    variant="primary",
                    elem_id="btn-ejecutar",
                )

            # ── COLUMNA DERECHA: Resultados ────────────────────────────────
            with gr.Column(scale=6):

                with gr.Tabs():

                    # ── Tab: Log ──────────────────────────────────────────────
                    with gr.TabItem("🖥️ Log del pipeline"):
                        log_out = gr.Textbox(
                            label="",
                            lines=24,
                            max_lines=40,
                            interactive=False,
                            elem_id="log-output",
                            placeholder="El log aparecerá aquí al ejecutar el pipeline…",
                        )

                    # ── Tab: Perfil ──────────────────────────────────────────
                    with gr.TabItem("📊 Perfil estadístico"):
                        perfil_out = gr.Dataframe(
                            label="Estadísticas por variable",
                            interactive=False,
                            wrap=True,
                        )

                    # ── Tab: Preview ─────────────────────────────────────────
                    with gr.TabItem("👁️ Vista previa"):
                        preview_out = gr.Dataframe(
                            label="Primeras 100 filas",
                            interactive=False,
                            wrap=False,
                        )

                    # ── Tab: Descarga ────────────────────────────────────────
                    with gr.TabItem("⬇️ Descargar"):
                        gr.Markdown(
                            "### Descargar Dataset\n"
                            "El archivo CSV incluye metadatos en las primeras líneas "
                            "(comentadas con `#`) y está listo para usar con scikit-learn:"
                        )
                        gr.Markdown("""
```python
import pandas as pd
df = pd.read_csv('dataset_ml_*.csv',
                 comment='#',
                 index_col='Timestamp',
                 parse_dates=True)
X = df.drop(columns=['PM25'])
y = df['PM25']
```
""")
                        csv_download = gr.File(
                            label="Dataset generado (CSV)",
                            interactive=False,
                        )

        # ── FOOTER ────────────────────────────────────────────────────────────
        gr.Markdown(
            "<br><center style='color:#94a3b8;font-size:0.78rem;'>"
            "REMMAQ · Red Metropolitana de Monitoreo Atmosférico de Quito · "
            "Dataset para HistGradientBoostingRegressor (scikit-learn)"
            "</center>"
        )

        # ══════════════════════════════════════════════════════════════════════
        # EVENTOS
        # ══════════════════════════════════════════════════════════════════════

        # Escanear directorio
        btn_scan.click(
            fn=escanear_directorio,
            inputs=[txt_dir],
            outputs=[status_box, parroquia_dd, tabla_archivos],
        ).then(lambda: gr.update(visible=True), outputs=[status_box])

        # Subida manual
        btn_upload.click(
            fn=procesar_subida,
            inputs=[file_upload],
            outputs=[status_box, parroquia_dd, tabla_archivos],
        ).then(lambda: gr.update(visible=True), outputs=[status_box])

        # Ejecutar pipeline
        btn_run.click(
            fn=ejecutar_pipeline,
            inputs=[parroquia_dd, excluir_pandemia, umbral_gap],
            outputs=[log_out, perfil_out, preview_out, csv_download],
        )

    return app


# ══════════════════════════════════════════════════════════════════════════════
# PUNTO DE ENTRADA
# ══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    app = build_app()
    app.launch(
        server_name="0.0.0.0",   # accesible en red local
        server_port=7860,
        share=False,             # True para un enlace público temporal de Gradio
        show_error=True,
        inbrowser=True,          # abre el navegador automáticamente
    )